In [ ]:
# ---------------------------------
# 0. Imports & reproducibility
# ---------------------------------
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

torch.manual_seed(42)
np.random.seed(42)


In [ ]:
logvar = torch.load("tr_logvar.pt")
mu = torch.load("tr_mu.pt")
labels = torch.load("tr_labels.pt")

# logvar = torch.load("data_1/tr_logvar.pt")
# mu = torch.load("data_1/tr_mu.pt")
# labels = torch.load("data_1/tr_labels.pt")
# v_logvar = torch.load("data_1/val_logvar.pt")
# v_mu = torch.load("data_1/val_mu.pt")
# v_labels = torch.load("data_1/val_labels.pt")
#
# logvar.extend(v_logvar)
# mu.extend(v_mu)
# labels.extend(v_labels)

### TOP

# logvar = logvar.detach().numpy()
# mu = mu.detach().numpy()
#
# print(logvar.shape)
# print(mu.shape)

# features = np.concatenate((logvar, mu), axis=1)
# features = mu

### SPLIT

print(len(logvar), len(logvar[0][0]))
print(len(mu), len(mu[0][0]))

features = []
for i in range(len(logvar)):
    l = logvar[i][0].detach().numpy()
    m = mu[i][0].detach().numpy()
    features.append(np.concatenate((l, m), axis=0))

features = np.array(features)

### BOTTOM

print(features.shape)

print(len(labels))

4801 128
4801 128
(4801, 256)
4801


In [ ]:
unique_labels = sorted(set(labels))
label_to_idx   = {lbl:idx for idx,lbl in enumerate(unique_labels)}
labels_idx     = [label_to_idx[l] for l in labels]
print(len(unique_labels))
print(len(labels_idx))

53
4801


In [ ]:

# ---------------------------------
# 1. Load/prepare the data
#    (assumes `features` & `labels_idx`
#     are already in memory)
# ---------------------------------
# features: (4801, 400)  float or int
# labels_idx: list[str]  length 4801

X = features.astype(np.float32)                         # (4801, 400)
y = np.array(labels_idx)                           # (4801,)


X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)  # 80 % train

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)  # 10 % / 10 %

# Torch tensors & loaders
batch_size = 64
train_dl = DataLoader(TensorDataset(torch.from_numpy(X_train),
                                    torch.from_numpy(y_train)),
                      batch_size=batch_size, shuffle=True)

valid_dl = DataLoader(TensorDataset(torch.from_numpy(X_valid),
                                    torch.from_numpy(y_valid)),
                      batch_size=batch_size)

test_dl  = DataLoader(TensorDataset(torch.from_numpy(X_test),
                                    torch.from_numpy(y_test)),
                      batch_size=batch_size)


In [ ]:
# ---------------------------------
# 2. Define a simple feed-forward NN
# ---------------------------------
class MLP(nn.Module):
    def __init__(self, d_in=256, d_out=53):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, d_out)        # logits for 53 classes
        )

    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)



In [ ]:
# ---------------------------------
# 3. Training loop
# ---------------------------------
num_epochs = 250
for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)

    # validation
    model.eval()
    with torch.no_grad():
        val_logits = torch.cat([model(xb.to(device)) for xb, _ in valid_dl])
        val_targets = torch.cat([yb for _, yb in valid_dl]).to(device)
        val_preds = val_logits.argmax(1)
        val_acc = (val_preds == val_targets).float().mean().item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d} • train loss {epoch_loss/len(train_dl.dataset):.4f} "
          f"• valid acc {val_acc*100:5.2f}%")



Epoch 10 • train loss 0.9095 • valid acc 73.54%
Epoch 20 • train loss 0.7421 • valid acc 78.33%
Epoch 30 • train loss 0.6630 • valid acc 78.54%
Epoch 40 • train loss 0.6089 • valid acc 78.75%
Epoch 50 • train loss 0.5572 • valid acc 79.58%
Epoch 60 • train loss 0.5294 • valid acc 82.08%
Epoch 70 • train loss 0.5029 • valid acc 81.46%
Epoch 80 • train loss 0.4741 • valid acc 81.88%
Epoch 90 • train loss 0.4563 • valid acc 81.25%
Epoch 100 • train loss 0.4276 • valid acc 82.71%
Epoch 110 • train loss 0.3951 • valid acc 82.50%
Epoch 120 • train loss 0.3932 • valid acc 81.25%
Epoch 130 • train loss 0.3658 • valid acc 81.46%
Epoch 140 • train loss 0.3570 • valid acc 81.67%
Epoch 150 • train loss 0.3361 • valid acc 82.71%
Epoch 160 • train loss 0.3256 • valid acc 81.88%
Epoch 170 • train loss 0.3145 • valid acc 81.46%
Epoch 180 • train loss 0.2894 • valid acc 82.29%
Epoch 190 • train loss 0.2912 • valid acc 82.08%
Epoch 200 • train loss 0.2800 • valid acc 81.25%
Epoch 210 • train loss 0.2630

In [ ]:
# torch.save(model, "MLPClassifier_new.pt")

In [ ]:
# ---------------------------------
# 4. Final evaluation on the hold-out test set
# ---------------------------------
model.eval()
with torch.no_grad():
    test_logits = torch.cat([model(xb.to(device)) for xb, _ in test_dl])
    test_targets = torch.cat([yb for _, yb in test_dl]).to(device)
    test_preds = test_logits.argmax(1)


test_acc = accuracy_score(test_targets.cpu(), test_preds.cpu())
print(f"\nTEST accuracy: {test_acc*100:5.2f}%")

sil_count = 0
sil_right = 0
other_count = 0
other_right = 0
true_sil = 0

sil_index = label_to_idx['sil']

y_true, y_pred = test_targets.cpu(), test_preds.cpu()

for i in range(len(y_pred)):
    if y_pred[i] == sil_index:
        sil_count += 1
        if y_pred[i] == y_true[i]:
            sil_right += 1
    else:
        other_count += 1
        if y_pred[i] == y_true[i]:
            other_right += 1
        elif y_true[i] == sil_index:
            true_sil += 1

print(f"Total 'sil' appearances: {sil_right + true_sil}/{sil_count + other_count} total words ({(sil_right + true_sil) / (sil_count + other_count) * 100:.1f}%)")
# print(f"Classifier 'sil' guesses: {sil_count}/{sil_count + other_count} ({(sil_count) / (sil_count + other_count) * 100:.1f}%)")
print(f"Guesses correct when guessing 'sil': {sil_right}/{sil_count} ({(sil_right) / (sil_count) * 100:.1f}%)")
print(f"Guesses correct when guessing other words: {other_right}/{other_count} ({(other_right) / (other_count) * 100:.1f}%)")
print(f"Times we guessed other words, when the true word was 'sil': {true_sil}/{other_count} ({(true_sil) / (other_count) * 100:.1f}%)")



TEST accuracy: 81.70%
Total 'sil' appearances: 120/481 total words (24.9%)
Guesses correct when guessing 'sil': 117/119 (98.3%)
Guesses correct when guessing other words: 276/362 (76.2%)
Times we guessed other words, when the true word was 'sil': 3/362 (0.8%)


In [ ]:
logvar = torch.load("test_logvar_new.pt")
mu = torch.load("test_mu_new.pt")
test_labels = torch.load("test_labels_new.pt")
# logvar = torch.load("data_1/test_logvar.pt")
# mu = torch.load("data_1/test_mu.pt")
# test_labels = torch.load("data_1/test_labels.pt")

### TOP

# logvar = logvar.detach().numpy()
# mu = mu.detach().numpy()
#
# print(logvar.shape)
# print(mu.shape)
#
# # test_features = np.concatenate((logvar, mu), axis=1)
# test_features = mu

### SPLIT

print(len(logvar), len(logvar[0][0]))
print(len(mu), len(mu[0][0]))

test_features = []
for i in range(len(logvar)):
    l = logvar[i][0].detach().numpy()
    m = mu[i][0].detach().numpy()
    test_features.append(np.concatenate((l, m), axis=0))

test_features = np.array(test_features)

### Bottom

print(test_features.shape)

unique_labels = sorted(set(test_labels))
label_to_idx   = {lbl:idx for idx,lbl in enumerate(unique_labels)}
test_labels_idx = [label_to_idx[l] for l in test_labels]

print(len(test_labels))
print(len(test_labels_idx))

new_X = test_features.astype(np.float32)                         # (4801, 400)
new_y = np.array(test_labels_idx)                           # (4801,)

batch_size = 64

new_test_dl  = DataLoader(TensorDataset(torch.from_numpy(new_X),
                                    torch.from_numpy(new_y)),
                      batch_size=batch_size)

1602 128
1602 128
(1602, 256)
1602
1602


In [ ]:
model.eval()
with torch.no_grad():
    test_logits = torch.cat([model(xb.to(device)) for xb, _ in new_test_dl])
    test_targets = torch.cat([yb for _, yb in new_test_dl]).to(device)
    test_preds = test_logits.argmax(1)

test_acc = accuracy_score(test_targets.cpu(), test_preds.cpu())
print(f"\nTEST accuracy: {test_acc*100:5.2f}%")

sil_count = 0
sil_right = 0
other_count = 0
other_right = 0
true_sil = 0

sil_index = label_to_idx['sil']

y_true, y_pred = test_targets.cpu(), test_preds.cpu()

for i in range(len(y_pred)):
    if y_pred[i] == sil_index:
        sil_count += 1
        if y_pred[i] == y_true[i]:
            sil_right += 1
    else:
        other_count += 1
        if y_pred[i] == y_true[i]:
            other_right += 1
        elif y_true[i] == sil_index:
            true_sil += 1

print(f"Total 'sil' appearances: {sil_right + true_sil}/{sil_count + other_count} total words ({(sil_right + true_sil) / (sil_count + other_count) * 100:.1f}%)")
# print(f"Classifier 'sil' guesses: {sil_count}/{sil_count + other_count} ({(sil_count) / (sil_count + other_count) * 100:.1f}%)")
print(f"Guesses correct when guessing 'sil': {sil_right}/{sil_count} ({(sil_right) / (sil_count) * 100:.1f}%)")
print(f"Guesses correct when guessing other words: {other_right}/{other_count} ({(other_right) / (other_count) * 100:.1f}%)")
print(f"Times we guessed other words, when the true word was 'sil': {true_sil}/{other_count} ({(true_sil) / (other_count) * 100:.1f}%)")


TEST accuracy: 82.71%
Total 'sil' appearances: 399/1602 total words (24.9%)
Guesses correct when guessing 'sil': 398/402 (99.0%)
Guesses correct when guessing other words: 927/1200 (77.2%)
Times we guessed other words, when the true word was 'sil': 1/1200 (0.1%)
